# Testing Navier Stokes solver.

1. Convert twoD -> threeD. DONE IN `TestingNavierStokes1.ipynb`
2. Add hydrostatic balance. DONE IN `TestingNavierStokes2.ipynb`
3. Add Coriolis force and customize to wind_forced_dishpan problem, except that the surface boundary condition is Dirichlet with a specified surface velocity.

Based on: `gridap` Tutorial 8: Incompressible Navier-Stokes

twnh July '25

## Problem statement

The goal is to solve a nonlinear multi-field PDE. Consider a well known benchmark in computational fluid dynamics, the lid-driven cavity for the incompressible rotating Navier-Stokes equations. Formally, the PDE we want to solve is: find the velocity vector $u$ and the pressure $p$ such that

$$
\left\lbrace
\begin{aligned}
-\nu  \nabla^2 u +  \frac{1}{\rho_0} \nabla p + g_r \hat{\mathbf k} + f  \hat{\mathbf k} \times u = 0 &\text{ in }\Omega,\\
\nabla\cdot u = 0 &\text{ in } \Omega,\\
u = u_0 &\text{ on } \partial\Omega,
\end{aligned}
\right.
$$

where the computational domain is the rectangle $\Omega \doteq (0,L_x) \times (-L_y/2,L_y/2) \times (-L_z,0)$. In this example, the driving force is the Dirichlet boundary velocity $u_0$, which is a non-zero velocity in the $x$ direction. Since we impose Dirichlet boundary conditions on the entire boundary $\partial\Omega$, the mean value of the pressure is constrained to equal zero,

$$
\int_\Omega p \ {\rm d}\Omega = 0.
$$


In [1]:
using Gridap

#### Define parameters

In [2]:

# The grid
Nx = 32             # number of points in x direction
Ny = 16             # number of points in y direction
Nz = 8              # number of points in the vertical direction

# The domain
Lx = 0.25            # (m) domain length
Ly = 0.125           # (m) domain width
Lz = 0.025           # (m) domain depth
domain = (0.0,Lx,-Ly/2,Ly/2,-Lz,0)
partition = (Nx,Ny,Nz)

# Create the mesh
model = CartesianDiscreteModel(domain,partition)

# Physical properties
gravity = 10.0      # (m/s^2)
nu = 1e-6           # (m^2/s) kinematic viscosity
rho0 = 1000.0       # (kg/m^3) reference density
# Rotation rate
rotation_period = 6.0   # (s)
f = 2*2*π/rotation_period
# f=0.0
# Surface speed
U0 = 2e-3  ;        # (m/s)

For convenience, we create two new boundary tags,  namely `"diri1"` and `"diri0"`, one for the top side of the square (where the velocity is non-zero), and another for the rest of the boundary (where the velocity is zero).

In [3]:
labels = get_face_labeling(model)
add_tag_from_tags!(labels,"diri1",[22,])
add_tag_from_tags!(labels,"diri0",[1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,23,24,25,26]) ;

## FE spaces

For the velocities, we need to create a conventional vector-valued continuous Lagrangian FE space. In this example, we select a second order interpolation.

In [4]:
D = 2
order = 2
reffeᵤ = ReferenceFE(lagrangian,VectorValue{3,Float64},order)
V = TestFESpace(model,reffeᵤ,conformity=:H1,labels=labels,dirichlet_tags=["diri0","diri1"])

UnconstrainedFESpace()

The interpolation space for the pressure is built as follows

In [5]:
reffeₚ = ReferenceFE(lagrangian,Float64,order-1;space=:P)
Q = TestFESpace(model,reffeₚ,conformity=:L2,constraint=:zeromean)

ZeroMeanFESpace()

With the options `:Lagrangian`, `space=:P`, `valuetype=Float64`, and `order=order-1`, we select the local polynomial space $P_{k-1}(T)$ on the cells $T\in\mathcal{T}$. With the symbol `space=:P` we specifically chose a local Lagrangian interpolation of type "P". Without using `space=:P`, would lead to a local Lagrangian of type "Q" since this is the default for quadrilateral or hexahedral elements. On the other hand, `constraint=:zeromean` leads to a FE space, whose functions are constrained to have mean value equal to zero, which is just what we need for the pressure space. With these objects, we build the trial multi-field FE spaces

In [6]:
uD0 = VectorValue(0,0,0)
uD2(x) = U0*VectorValue(cos(π*x[2]/Ly),0,0) # Add forcing with curl here.
U = TrialFESpace(V,[uD0,uD2])
P = TrialFESpace(Q)
Y = MultiFieldFESpace([V, Q])
X = MultiFieldFESpace([U, P])

MultiFieldFESpace()

## Triangulation and integration quadrature

From the discrete model we can define the triangulation and integration measure

In [7]:
degree = order
Ωₕ = Triangulation(model)
dΩ = Measure(Ωₕ,degree)

GenericMeasure()

The bilinear form reads

In [ ]:
using LinearAlgebra
khat = VectorValue(0,0,1)
a((u,p),(v,q)) = ∫( nu*∇(v)⊙∇(u) - (1/rho0)*(∇⋅v)*p + (1/rho0)*q*(∇⋅u) + gravity*v⋅khat + f*(cross(khat, v)⋅u) + f*(cross(khat, u)⋅v))dΩ

a (generic function with 1 method)

Finally, the Navier-Stokes weak form residual is defined as

In [9]:
res((u,p),(v,q)) = a((u,p),(v,q))

res (generic function with 1 method)

With the function `res` representing the weak residual, we build the nonlinear FE problem:

In [10]:
op = FEOperator(res,X,Y)

FEOperatorFromWeakForm()

## Nonlinear solver phase

In [11]:
using LineSearches: BackTracking
nls = NLSolver(show_trace=true, method=:newton, linesearch=BackTracking())
solver = FESolver(nls)

NonlinearFESolver()

Solve the problem with a random initial guess if it's the first time, otherwise, use the previous solution. This allows parameter changes to be explored, but not different resolutions.

In [12]:
import Random
Random.seed!(1234)
using Gridap.MultiField

# Generate random data for the free degrees of freedom
x_U = rand(Float64, num_free_dofs(U))
U0 = FEFunction(U, x_U)
x_P = rand(Float64, num_free_dofs(P))
P0 = FEFunction(P, x_P)

# Create a MultiFieldFESpace from U and P
multi_field_space = MultiFieldFESpace([U, P])

# Create the MultiFieldFEFunction using the three arguments required by the constructor
X0 = MultiFieldFEFunction([x_U; x_P], multi_field_space, [U0, P0]) ;

In [13]:
if @isdefined(cache)
    @time X0,cache = solve!(X0,solver,op,cache)
else
    @time X0,cache = solve!(X0,solver,op)
end

# Accessing the fields within the MultiFieldFEFunction
uh = X0.single_fe_functions[1]
ph = X0.single_fe_functions[2] ;

Iter     f(x) inf-norm    Step 2-norm 
------   --------------   --------------
     0     6.082232e-07              NaN
     1     7.409024e-18     3.738453e+08
 36.752064 seconds (57.66 M allocations: 11.211 GiB, 1.62% gc time, 46.24% compilation time)


Finally, we write the results for visualization.

In [14]:
writevtk(Ωₕ,"TestingNavierStokes3",cellfields=["uh"=>uh,"ph"=>ph])

(["TestingNavierStokes3.vtu"],)